In [9]:
import warnings
warnings.filterwarnings('ignore')

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from utils import *

from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    average_precision_score,
    recall_score,
    precision_score,
    f1_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
import xgboost as xgb

from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split, cross_validate
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [11]:
pd.set_option('display.max_colwidth', None)

In [12]:
COLUNA_ALVO = 'Class'
RANDOM_STATE = 42

In [13]:
df = pd.read_csv('../data/creditcard.csv', delimiter=',')
print("Shape:", df.shape)
print("Columns:", df.columns)

X = df.drop(COLUNA_ALVO, axis=1)
y = df[COLUNA_ALVO]

Shape: (284807, 31)
Columns: Index(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
       'Class'],
      dtype='object')


In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Grids e modelos

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), ['Time', 'Amount'])
    ],
    remainder='passthrough'
)

In [16]:
pipeline_rfc_undersampling = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('sampler', RandomUnderSampler(random_state=RANDOM_STATE)),
    ('model', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

pipeline_lr_undersampling = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('sampler', RandomUnderSampler(random_state=RANDOM_STATE)),
    ('model', LogisticRegression(random_state=RANDOM_STATE, max_iter=500, n_jobs=-1))
])

pipeline_knn_undersampling = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('sampler', RandomUnderSampler(random_state=RANDOM_STATE)),
    ('model', KNeighborsClassifier(n_jobs=-1))
])

pipeline_xgb_undersampling = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('sampler', RandomUnderSampler(random_state=RANDOM_STATE)),
    ('model', xgb.XGBClassifier(seed=RANDOM_STATE, n_jobs=-1))
])

pipeline_nb_undersampling = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('sampler', RandomUnderSampler(random_state=RANDOM_STATE)),
    ('model', GaussianNB())
])

In [17]:
param_grid_rfc = {
    'model__n_estimators': [30, 100, 300, 500],
    'model__max_depth': [7, 10, None],
    'model__min_samples_leaf': [1, 5, None]
}

param_grid_lr = {
    'model__penalty': ['l1', 'l2'],
    'model__C': [0.01, 0.1, 1.0],
    'model__solver': ['liblinear', 'saga']
}

param_grid_knn = {
    'model__n_neighbors': [3, 5, 7],
    'model__weights': ['uniform', 'distance']
}

param_grid_xgb = {
    "model__booster": ['gbtree', 'dart'],
    "model__max_depth": [5, 10, 50, 100],
    "model__learning_rate": [0.1, 0.01, 0.05],
    "model__n_estimators": [30, 100, 300, 500],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

param_grid_nb = {
    'model__var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]
}

In [18]:
models_to_run_undersampling = {
    'NaiveBayes': {
        'estimator': pipeline_nb_undersampling,
        'param_grid': param_grid_nb
    },
    'RandomForest': {
        'estimator': pipeline_rfc_undersampling,
        'param_grid': param_grid_rfc
    },
    'LogisticRegression': {
        'estimator': pipeline_lr_undersampling,
        'param_grid': param_grid_lr
    },
    'KNN': {
        'estimator': pipeline_knn_undersampling,
        'param_grid': param_grid_knn
    },
    'XGBoost': {
        'estimator': pipeline_xgb_undersampling,
        'param_grid': param_grid_xgb
    }
}

In [19]:
def run_single_model_nested_cv(
    estimator,
    param_grid,
    X,
    y,
    scoring=['accuracy', 'recall', 'precision', 'f1', 'roc_auc'],
    n_splits_outer=5,
    n_splits_inner=5,
    scoring_grid_search='f1',
    random_state=RANDOM_STATE):
    
    cv_inner = StratifiedKFold(n_splits=n_splits_inner, shuffle=True, random_state=random_state)
    cv_outer = StratifiedKFold(n_splits=n_splits_outer, shuffle=True, random_state=random_state)

    search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring=scoring_grid_search,
        cv=cv_inner,
        refit=True,
        n_jobs=1
    )

    output_ncv = cross_validate(
        estimator=search,
        X=X,
        y=y,
        scoring=scoring,
        cv=cv_outer,
        n_jobs=-1,
        return_estimator=True,
        return_train_score=True
    )

    df_results = pd.DataFrame(output_ncv)

    metrics_cols = [col for col in df_results.columns if col.startswith('test_')]
    summary = df_results[metrics_cols].agg(['mean', 'std']).round(3)

    return df_results, summary
    

In [ ]:
all_results = []
all_summaries = []

for model_name, config in models_to_run_undersampling.items():
    print(f"Rodando modelo: {model_name}")
    
    df_model_results, df_model_summary = run_single_model_nested_cv(
        estimator=config['estimator'],
        param_grid=config['param_grid'],
        X=X_train,
        y=y_train
    )
    
    df_model_results['model'] = model_name
    df_model_summary['model'] = model_name

    all_results.append(df_model_results)
    all_summaries.append(df_model_summary)

final_results_df = pd.concat(all_results, ignore_index=True)
final_results_df['best_params'] = final_results_df['estimator'].apply(lambda est: est.best_params_)
final_summary_df = pd.concat(all_summaries, keys=[key for key in models_to_run_undersampling.keys()])

Rodando modelo: NaiveBayes


Rodando modelo: RandomForest


In [ ]:
display(final_results_df)

,fit_time,score_time,estimator,test_accuracy,train_accuracy,test_recall,train_recall,test_precision,train_precision,test_f1,train_f1,test_roc_auc,train_roc_auc,model,best_params
0,162.288244,1.384839,"GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),\n estimator=Pipeline(steps=[('preprocessor',\n ColumnTransformer(remainder='passthrough',\n transformers=[('scaler',\n StandardScaler(),\n ['Time',\n 'Amount'])])),\n ('sampler',\n RandomUnderSampler(random_state=42)),\n ('model',\n RandomForestClassifier(n_jobs=-1,\n random_state=42))]),\n n_jobs=1,\n param_grid={'model__max_depth': [7, 10, None],\n 'model__min_samples_leaf': [1, 5, None],\n 'model__n_estimators': [30, 100, 300, 500]},\n scoring='f1')",0.975488,0.976739,0.923077,0.936709,0.060862,0.065545,0.114195,0.122517,0.987449,0.993264,RandomForest,"{'model__max_depth': 7, 'model__min_samples_leaf': 5, 'model__n_estimators': 300}"
1,160.782858,0.773339,"GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),\n estimator=Pipeline(steps=[('preprocessor',\n ColumnTransformer(remainder='passthrough',\n transformers=[('scaler',\n StandardScaler(),\n ['Time',\n 'Amount'])])),\n ('sampler',\n RandomUnderSampler(random_state=42)),\n ('model',\n RandomForestClassifier(n_jobs=-1,\n random_state=42))]),\n n_jobs=1,\n param_grid={'model__max_depth': [7, 10, None],\n 'model__min_samples_leaf': [1, 5, None],\n 'model__n_estimators': [30, 100, 300, 500]},\n scoring='f1')",0.976629,0.977002,0.848101,0.930159,0.059821,0.065651,0.111760,0.122645,0.974008,0.993240,RandomForest,"{'model__max_depth': 7, 'model__min_samples_leaf': 5, 'model__n_estimators': 300}"
2,157.002174,0.490366,"GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),\n estimator=Pipeline(steps=[('preprocessor',\n ColumnTransformer(remainder='passthrough',\n transformers=[('scaler',\n StandardScaler(),\n ['Time',\n 'Amount'])])),\n ('sampler',\n RandomUnderSampler(random_state=42)),\n ('model',\n RandomForestClassifier(n_jobs=-1,\n random_state=42))]),\n n_jobs=1,\n param_grid={'model__max_depth': [7, 10, None],\n 'model__min_samples_leaf': [1, 5, None],\n 'model__n_estimators': [30, 100, 300, 500]},\n scoring='f1')",0.980645,0.980820,0.860759,0.930159,0.072417,0.077781,0.133595,0.143557,0.972876,0.994360,RandomForest,"{'model__max_depth': 7, 'model__min_samples_leaf': 5, 'model__n_estimators': 100}"
3,161.304487,0.844287,"GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),\n estimator=Pipeline(steps=[('preprocessor',\n ColumnTransformer(remainder='passthrough',\n transformers=[('scaler',\n StandardScaler(),\n ['Time',\n 'Amount'])])),\n ('sampler',\n RandomUnderSampler(random_state=42)),\n ('model',\n RandomForestClassifier(n_jobs=-1,\n random_state=42))]),\n n_jobs=1,\n param_grid={'model__max_depth': [7, 10, None],\n 'model__min_samples_leaf': [1, 5, None],\n 'model__n_estimators': [30, 100, 300, 500]},\n scoring='f1')",0.976431,0.976931,0.936709,0.936508,0.064742,0.065848,0.121113,0.123045,0.988386,0.993994,RandomForest,"{'model__max_depth': 7, 'model__min_samples_leaf': 5, 'model__n_estimators': 300}"
4,160.191773,0.271893,"GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),\n estimator=Pipeline(steps=[('preprocessor',\n ColumnTransformer(remainder='passthrough',\n transformers=[('scaler',\n StandardScaler(),\n ['Time',\n 'Amount'])])),\n ('sampler',\n RandomUnderSampler(random_state=42)),\n ('model',\n RandomForestClassifier(n_jobs=-1,\n random_state=42))]),\n n_jobs=1,\n param_grid={'model__max_depth': [7, 10, None],\n 'model__min_samples_leaf': [1, 5, None],\n 'model__n_estimators': [30, 100, 300, 500]},\n scoring='f1')",0.975992,0.975351,0.886076,0.946032,0.060606,0.062421,0.113452,0.117115,0.969680,0.993593,RandomForest,"{'model__max_depth': 7, 'model__min_samples_leaf': 5, 'model__n_estimators': 100}"


In [ ]:
display(final_summary_df)

test_accuracy  test_recall  test_precision  test_f1  \
RandomForest mean          0.977        0.891           0.064    0.119   
             std           0.002        0.038           0.005    0.009   

                   test_roc_auc         model  
RandomForest mean         0.978  RandomForest  
             std          0.009  RandomForest